# Feature Engineering & Preprocessing Pipeline
RabTech Academy — AI & Machine Learning | Task 03

This notebook builds a reusable, leakage-safe preprocessing pipeline for the provided customer churn training dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


## 1. Load and inspect the dataset


In [ ]:
df = pd.read_csv('customer-churn-training.csv')
print('Shape:', df.shape)
display(df.head())
print('\nMissing values:\n', df.isna().sum())


## 2. Separate target and features
`customer_id` is an identifier and is excluded from modeling. The target is `churned`.

In [ ]:
target = 'churned'
drop_cols = ['customer_id']
X = df.drop(columns=[target] + drop_cols)
y = df[target]

numeric_features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
categorical_features = ['plan_type']

print('Numeric:', numeric_features)
print('Categorical:', categorical_features)


## 3. Train/test split before transformations
The split is performed before fitting imputers, scalers, or encoders. This prevents information from the test set leaking into the preprocessing steps.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


## 4. Build preprocessing pipelines
Numeric columns use median imputation followed by standard scaling. The categorical column uses most-frequent imputation followed by one-hot encoding.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
], remainder='drop')


## 5. Fit only on training data and transform both sets


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)


## 6. Verify encoded feature names


In [ ]:
feature_names = preprocessor.get_feature_names_out()
print('Processed features:')
for name in feature_names:
    print('-', name)


## 7. Correlation analysis
For a simple correlation view, categorical `plan_type` is represented using one-hot columns. Correlations here are exploratory and should not be interpreted as causal relationships.

In [ ]:
corr_df = pd.get_dummies(df.drop(columns=['customer_id']), columns=['plan_type'], dtype=int)
correlations = corr_df.corr(numeric_only=True)['churned'].sort_values(ascending=False)
print(correlations)


## 8. Leakage and reproducibility checks
- The identifier `customer_id` is excluded.
- Train/test splitting occurs before preprocessing is fitted.
- Imputation, scaling, and encoding are contained inside a Scikit-Learn pipeline.
- `handle_unknown='ignore'` prevents unseen categorical levels in the test/deployment data from breaking the transformer.
- The supplied dataset is extremely small (12 records), so this notebook demonstrates the pipeline rather than providing reliable production performance estimates.

## Conclusion
A reusable preprocessing pipeline was created using `ColumnTransformer` and `Pipeline`. Numeric variables are imputed and scaled, while the categorical plan variable is imputed and one-hot encoded. The train/test split occurs before fitting transformations, reducing the risk of data leakage. Correlation analysis is included as an exploratory diagnostic, and the small sample size is documented as a limitation.